In [221]:
# ============================================================
# CELL 0: SETUP & DATA LOADING
# ============================================================

# Import pandas library for data manipulation
import pandas as pd
# Import sqlite3 library for database operations
import sqlite3

# Configure pandas to display all columns without truncation
pd.set_option('display.max_columns', None)

# Load the CSV file into a DataFrame
df = pd.read_csv('telco.csv')

# Convert the DataFrame into a SQL table in the database
# if_exists='replace' means overwrite if the table already exists
conn = sqlite3.connect('telecom_churn.db')
df.to_sql('churn_data', conn, if_exists='replace', index=False)


print("\n✓ Database setup complete!")



✓ Database setup complete!


In [222]:

# ============================================================
# CELL 1: QUERY FUNCTION
# ============================================================

# Define a reusable function to execute SQL queries
# This function takes a SQL query string as input
def run_query(query):
    return pd.read_sql_query(query, conn)

In [223]:

# ============================================================
# CELL 2: VIEW SAMPLE DATA
# ============================================================

query = """
SELECT * 
FROM churn_data 
LIMIT 5;
"""

run_query(query)

,Customer ID,Gender,Age,Under 30,Senior Citizen,Married,Dependents,Number of Dependents,Country,State,City,Zip Code,Latitude,Longitude,Population,Quarter,Referred a Friend,Number of Referrals,Tenure in Months,Offer,Phone Service,Avg Monthly Long Distance Charges,Multiple Lines,Internet Service,Internet Type,Avg Monthly GB Download,Online Security,Online Backup,Device Protection Plan,Premium Tech Support,Streaming TV,Streaming Movies,Streaming Music,Unlimited Data,Contract,Paperless Billing,Payment Method,Monthly Charge,Total Charges,Total Refunds,Total Extra Data Charges,Total Long Distance Charges,Total Revenue,Satisfaction Score,Customer Status,Churn Label,Churn Score,CLTV,Churn Category,Churn Reason
0,8779-QRDMV,Male,78,No,Yes,No,No,0,United States,California,Los Angeles,90022,34.023810,-118.156582,68701,Q3,No,0,1,NaN,No,0.00,No,Yes,DSL,8,No,No,Yes,No,No,Yes,No,No,Month-to-Month,Yes,Bank Withdrawal,39.65,39.65,0.00,20,0.00,59.65,3,Churned,Yes,91,5433,Competitor,Competitor offered more data
1,7495-OOKFY,Female,74,No,Yes,Yes,Yes,1,United States,California,Los Angeles,90063,34.044271,-118.185237,55668,Q3,Yes,1,8,Offer E,Yes,48.85,Yes,Yes,Fiber Optic,17,No,Yes,No,No,No,No,No,Yes,Month-to-Month,Yes,Credit Card,80.65,633.30,0.00,0,390.80,1024.10,3,Churned,Yes,69,5302,Competitor,Competitor made better offer
2,1658-BYGOY,Male,71,No,Yes,No,Yes,3,United States,California,Los Angeles,90065,34.108833,-118.229715,47534,Q3,No,0,18,Offer D,Yes,11.33,Yes,Yes,Fiber Optic,52,No,No,No,No,Yes,Yes,Yes,Yes,Month-to-Month,Yes,Bank Withdrawal,95.45,1752.55,45.61,0,203.94,1910.88,2,Churned,Yes,81,3179,Competitor,Competitor made better offer
3,4598-XLKNJ,Female,78,No,Yes,Yes,Yes,1,United States,California,Inglewood,90303,33.936291,-118.332639,27778,Q3,Yes,1,25,Offer C,Yes,19.76,No,Yes,Fiber Optic,12,No,Yes,Yes,No,Yes,Yes,No,Yes,Month-to-Month,Yes,Bank Withdrawal,98.50,2514.50,13.43,0,494.00,2995.07,2,Churned,Yes,88,5337,Dissatisfaction,Limited range of services
4,4846-WHAFZ,Female,80,No,Yes,Yes,Yes,1,United States,California,Whittier,90602,33.972119,-118.020188,26265,Q3,Yes,1,37,Offer C,Yes,6.33,Yes,Yes,Fiber Optic,14,No,No,No,No,No,No,No,Yes,Month-to-Month,Yes,Bank Withdrawal,76.50,2868.15,0.00,0,234.21,3102.36,2,Churned,Yes,67,2793,Price,Extra data charges


In [224]:
# ============================================================
# CELL 3: CUSTOMER STATUS OVERVIEW
# ============================================================

# Get count of customers by their status (Churned, Stayed, Joined)
# This helps understand the overall churn situation
query = """
SELECT [Customer Status], 
       COUNT(*) AS Total_Customers
FROM churn_data
WHERE [Internet Type] IS NOT NULL
GROUP BY [Customer Status];
"""
run_query(query)

,Customer Status,Total_Customers
0,Churned,1756
1,Joined,272
2,Stayed,3489


In [225]:
# ============================================================
# CELL 4: CONTRACT TYPE IMPACT ON CHURN
# ============================================================

# Analyze churn rate by contract type
# Shows how many customers churned vs stayed for each contract duration
query = """
SELECT Contract, [Customer Status], COUNT(*) AS Total_Customers
FROM churn_data
WHERE [Customer Status] IN ('Churned', 'Stayed')
GROUP BY Contract, [Customer Status]
ORDER BY Contract;
"""
run_query(query)

,Contract,Customer Status,Total_Customers
0,Month-to-Month,Churned,1655
1,Month-to-Month,Stayed,1547
2,One Year,Churned,166
3,One Year,Stayed,1360
4,Two Year,Churned,48
5,Two Year,Stayed,1813


In [226]:
# ============================================================
# CELL 5: TOP REASONS FOR CHURN
# ============================================================

# Identify the top 5 reasons why customers churned
# Helps understand the main pain points and dissatisfaction factors
query = """ 
SELECT [Churn Reason], COUNT(*) AS Total_Customers
FROM churn_data
WHERE [Customer Status] = 'Churned'
GROUP BY [Churn Reason]
ORDER BY Total_Customers DESC
LIMIT 5;
"""
run_query(query)

,Churn Reason,Total_Customers
0,Competitor had better devices,313
1,Competitor made better offer,311
2,Attitude of support person,220
3,Don't know,130
4,Competitor offered more data,117


In [227]:
# ============================================================
# CELL 6: INTERNET TYPE IMPACT ON CHURN
# ============================================================

# Compare churn count by internet type (DSL, Cable, Fiber Optic)
# Shows which internet type has highest customer churn
query = """
SELECT [Internet Type], [Customer Status], COUNT(*) AS Total_Cust
FROM churn_data
WHERE [Customer Status] = 'Churned'
GROUP BY [Internet Type], [Customer Status]  
ORDER BY COUNT(*) DESC;
"""
run_query(query)

,Internet Type,Customer Status,Total_Cust
0,Fiber Optic,Churned,1236
1,DSL,Churned,307
2,Cable,Churned,213
3,NaN,Churned,113


In [228]:
# ============================================================
# CELL 7: REVENUE IMPACT BY INTERNET TYPE
# ============================================================

# Calculate average monthly charge and annualized revenue loss by internet type
# Identifies which service generates highest revenue but loses most customers
query = """ 
SELECT [Internet Type], 
       COUNT(*) AS Total_Customers,
       ROUND(SUM([Monthly Charge]) * 12, 0) AS Annual_Revenue_Loss
FROM churn_data
WHERE [Customer Status] = 'Churned'
GROUP BY [Internet Type]
ORDER BY Annual_Revenue_Loss DESC;
"""
run_query(query)

,Internet Type,Total_Customers,Annual_Revenue_Loss
0,Fiber Optic,1236,1305797.0
1,DSL,307,180529.0
2,Cable,213,155625.0
3,NaN,113,27619.0
